In this notebook, a synthetic version of the original clinical dataset is generated so there can be reproducibility of the work. The synthetic data are created using a $\textbf{Gaussian Copula-based approach}$, with the objective of maintaining the statistical properties of the original dataset, including variable distributions and dependencies between variables.

I also takes care of all the different data types, including $\textbf{numerical, categorical, and one-hot encoded variables}$. One-hot encoded groups are temporarily reconstructed into their original categorical representation during the synthesis process to ensure that valid category combinations are preserved. After generation, the synthetic dataset is transformed back to the original format so that it retains the same structure and columns as the real dataset.

Finally, several validation procedures are performed to assess the similarity between the real and synthetic datasets, including comparisons of variable distributions, categorical frequencies, and relationships between variables.

# Imports and full procedure and pipeline

In [1]:
# %pip install sdv

In [2]:
import numpy as np
import pandas as pd

import os

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

from scipy.stats import ks_2samp, wasserstein_distance

import warnings
warnings.filterwarnings("ignore")

In [3]:
path_data = "dataframes/df_ready.csv"
df = pd.read_csv(path_data)

OUTPUT_PATH = "dataframes/synthetic_clinical_data.csv"

In [4]:
TARGET_COL = "VTE"
TARGET_TYPE = "categorical"
N_SYNTHETIC_ROWS = None

## Define columns

In [5]:
# DEFINE COLUMN GROUPS

numeric_cols = [
    "age_cancer_dx", "bmi"
]

binary_cat_cols = [
    "chronic_cardiac_insufficiency", "copd", "renal_insufficiency", "arterial_hypertension",
    "diabetes_mellitus", "dyslipidemia", "hormone_therapy", "previous_vte", "previous_ate",
    "venous_insufficiency", "family_background_vte", "tumor_surgically_removed", "catheter_device",
    "sex"
]

discrete_cat_cols = [
    "pTNM_stage", "performance_status", "grade_histological_differentiation"
]


tobacco_ohe = [
    "tobacco_use_Active smoker", "tobacco_use_Former smoker", "tobacco_use_Never has smoked"
]

tumor_ohe = [
    "primary_tumor_Bile duct", "primary_tumor_Brain", "primary_tumor_Colorectal", "primary_tumor_Gastric",
    "primary_tumor_NSCLC", "primary_tumor_Oesophageal", "primary_tumor_Pancreatic"
]

tobacco_categories = [
    "Active smoker", "Former smoker", "Never has smoked"
]

tumor_categories = [
    "Bile duct", "Brain", "Colorectal", "Gastric", "NSCLC", "Oesophageal", "Pancreatic"
]

## Helper functions

In [6]:
def collapse_onehot_strict(df, cols, new_col, drop_original=True):
    """
    Converts a one‑hot encoded group into a single categorical column.
    It only accepts rows with exactly one value equal to 1; otherwise, it assigns NaN.
    """
    out = df.copy()
    subset = out[cols].fillna(0)

    # Sum of 1s per row
    row_sum = subset.sum(axis=1)

    # Index of the winning category
    idx = subset.to_numpy().argmax(axis=1)

    # Extract category names from the prefix
    labels = [c.replace(new_col + "_", "") for c in cols]

    # Assign category only if there is exactly one 1
    out[new_col] = np.where(
        row_sum == 1,
        np.array(labels, dtype=object)[idx],
        np.nan
    )

    if drop_original:
        out = out.drop(columns=cols)
        
    return out


In [7]:
def expand_onehot(df, col, categories, prefix):
    """
    Converts a categorical column into a one‑hot encoded representation with fixed columns.
    Ensures that the expected columns are always produced.
    """
    out = df.copy()

    # Force the expected set of categories
    out[col] = pd.Categorical(out[col], categories=categories)

    # One-hot
    dummies = pd.get_dummies(out[col], prefix=prefix)

    # Ensure that all expected columns are present
    expected = [f"{prefix}_{cat}" for cat in categories]
    for c in expected:
        if c not in dummies.columns:
            dummies[c] = 0

    dummies = dummies[expected].astype(int)
    out = pd.concat([out.drop(columns=[col]), dummies], axis=1)
    
    return out

In [8]:
def force_categorical_dtype(df, cols):
    """
    Mark columns as object so that SDV treats them as categorical
    """
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype("object")
    return out

In [9]:
def cast_discrete_back_to_int(df, cols):
    """
    Convert discrete categorical columns returned by the synthesizer into 
    integers when their values correspond to 1/2/3/4, etc.
    """
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
    return out

In [10]:
def cast_binary_back_to_int(df, cols):
    """
    Convert binary categorical columns into integer values 0/1
    """
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")
    return out

In [11]:
def compare_basic_stats(real_df, syn_df, numeric_cols, categorical_cols):
    """
    Compare means/standard deviations for numerical variables and proportions for categorical variables.
    Returns:
    ‑ a numerical summary table
    ‑ a dictionary of categorical frequency tables
    """
    num_summary = []
    for c in numeric_cols:
        if c in real_df.columns and c in syn_df.columns:
            num_summary.append({
                "column": c,
                "real_mean": real_df[c].mean(),
                "syn_mean": syn_df[c].mean(),
                "real_std": real_df[c].std(),
                "syn_std": syn_df[c].std(),
            })

    cat_summary = {}
    for c in categorical_cols:
        if c in real_df.columns and c in syn_df.columns:
            cat_summary[c] = pd.concat(
                [
                    real_df[c].value_counts(normalize=True, dropna=False).rename("real"),
                    syn_df[c].value_counts(normalize=True, dropna=False).rename("synthetic"),
                ],
                axis=1
            ).fillna(0)

    return pd.DataFrame(num_summary), cat_summary

## Load data

In [12]:
# Automatically detect *_n_risk_alleles
risk_allele_cols = [c for c in df.columns if c.endswith("_n_risk_alleles")]
numeric_cols = numeric_cols + risk_allele_cols

if N_SYNTHETIC_ROWS is None:
    N_SYNTHETIC_ROWS = len(df)

## Preprocess real data

In [13]:
df2 = df.copy()

# Collapse one-hot groups into a single categorical column
df2 = collapse_onehot_strict(df2, tobacco_ohe, "tobacco_use")
df2 = collapse_onehot_strict(df2, tumor_ohe, "primary_tumor")

# Final set of categorical variables for synthesis
categorical_cols = (
    binary_cat_cols
    + discrete_cat_cols
    + ["tobacco_use", "primary_tumor"]
)

# Force categorical dtype even if encoded as numbers
df2 = force_categorical_dtype(df2, categorical_cols)

# Columns used in the synthesis
cols_for_synthesis = numeric_cols + categorical_cols + [TARGET_COL]

# Filter only the necessary columns
df_model = df2[cols_for_synthesis].copy()

## Build SDV Metadata

In [14]:
# Force real numeric columns
for col in numeric_cols:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# If the target is numerical, convert it as well
if TARGET_TYPE == "numerical":
    df_model[TARGET_COL] = pd.to_numeric(df_model[TARGET_COL], errors="coerce")

# Force categorical columns
for col in categorical_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype("object")

In [15]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_model)

# Enforce correct types
for col in numeric_cols:
    metadata.update_column(col, sdtype="numerical")

for col in categorical_cols:
    metadata.update_column(col, sdtype="categorical")

metadata.update_column(TARGET_COL, sdtype=TARGET_TYPE)

## Build synthesizer

In [16]:
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df_model)

## Samples synthetic data

In [17]:
synthetic_raw = synthesizer.sample(num_rows=N_SYNTHETIC_ROWS)

## Postprocess synthetic data

In [18]:
synthetic_df = synthetic_raw.copy()

# Convert discrete variables back to integers if you want to preserve the original format
synthetic_df = cast_discrete_back_to_int(
    synthetic_df,
    ["pTNM_stage", "performance_status", "grade_histological_differentiation"]
)

# Convert binary variables back to integer 0/1
synthetic_df = cast_binary_back_to_int(synthetic_df, binary_cat_cols)

# If the target is categorical and in your original dataset it was numerically encoded,
# you can convert it back to integer
if TARGET_TYPE == "categorical":
    synthetic_df[TARGET_COL] = pd.to_numeric(
        synthetic_df[TARGET_COL], errors="ignore"
    )

# Re-expand one-hot if your model expects the same input as the original dataset
synthetic_df = expand_onehot(
    synthetic_df,
    "tobacco_use",
    tobacco_categories,
    "tobacco_use"
)

synthetic_df = expand_onehot(
    synthetic_df,
    "primary_tumor",
    tumor_categories,
    "primary_tumor"
)

## Reorder columns to amtch original dataset

In [19]:
# Keep the same column order as the original dataset,
# as long as the columns exist in the synthetic one.

final_cols = [c for c in df.columns if c in synthetic_df.columns]
synthetic_df = synthetic_df[final_cols]

# Save

In [20]:
synthetic_df.to_csv(OUTPUT_PATH, index=False)

print("Synthetic dataset saved to:", OUTPUT_PATH)
print("Real shape:      ", df.shape)
print("Synthetic shape: ", synthetic_df.shape)

Synthetic dataset saved to: dataframes/synthetic_clinical_data.csv
Real shape:       (391, 40)
Synthetic shape:  (391, 40)


# Basic checks

In [21]:
# To compare statistics you must use the pre‑synthesis dataset (df_model / synthetic_raw),
# because there tobacco_use and primary_tumor are still single categorical columns.
num_cmp, cat_cmp = compare_basic_stats(
    real_df=df_model,
    syn_df=synthetic_raw,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols + [TARGET_COL] if TARGET_TYPE == "categorical" else categorical_cols
)

In [22]:
print("\n=== Numeric summary ===")
print(num_cmp)

print("\n=== Categorical summaries ===")
for c in ["pTNM_stage", "performance_status", "tobacco_use", "primary_tumor"]:
    if c in cat_cmp:
        print(f"\n--- {c} ---")
        print(cat_cmp[c])


=== Numeric summary ===
                       column  real_mean   syn_mean   real_std    syn_std
0               age_cancer_dx  64.797954  65.189258  10.991026  10.694354
1                         bmi  24.921637  25.061586   4.175036   4.374070
2   rs11696364_n_risk_alleles   0.199488   0.872123   0.448473   0.890989
3     rs169713_n_risk_alleles   0.539642   1.168798   0.626622   0.724762
4       rs4524_n_risk_alleles   1.483376   1.462916   0.623559   0.642935
5       rs6003_n_risk_alleles   1.813299   1.695652   0.403100   0.625900
6    rs2227631_n_risk_alleles   0.928389   1.491049   0.690587   0.563229
7        rs268_n_risk_alleles   0.033248   1.000000   0.179513   0.000000
8       rs5110_n_risk_alleles   0.171355   0.969309   0.409875   0.873589
9       rs6025_n_risk_alleles   0.028133   1.000000   0.165565   0.000000
10   rs2232698_n_risk_alleles   0.023018   1.000000   0.150152   0.000000
11      rs5985_n_risk_alleles   0.450128   1.219949   0.600806   0.766418

=== Categori

## More checks

In [23]:
def structural_checks(real_df, syn_df):
    """
    Check columns, missing values, and general types.
    """
    real_cols = set(real_df.columns)
    syn_cols = set(syn_df.columns)

    print("Missing in synthetic:", sorted(real_cols - syn_cols))
    print("Extra in synthetic:   ", sorted(syn_cols - real_cols))
    print("\nReal shape:", real_df.shape)
    print("Synthetic shape:", syn_df.shape)
    print("\nMissings in real:")
    print(real_df.isna().sum()[real_df.isna().sum() > 0].sort_values(ascending=False))
    print("\nMissings in synthetic:")
    print(syn_df.isna().sum()[syn_df.isna().sum() > 0].sort_values(ascending=False))

In [24]:
def check_onehot_validity(df, cols, group_name):
    """
    Check whether each row has exactly one 1 in the one‑hot group.
    """
    row_sum = df[cols].sum(axis=1)
    print(f"\n[{group_name}]")
    print("Rows with exactly one 1:", (row_sum == 1).mean())
    print("Rows with zero 1s:      ", (row_sum == 0).mean())
    print("Rows with >1 1s:        ", (row_sum > 1).mean())

In [25]:
structural_checks(df, synthetic_df)
check_onehot_validity(synthetic_df, tobacco_ohe, "tobacco_use")
check_onehot_validity(synthetic_df, tumor_ohe, "primary_tumor")

Missing in synthetic: []
Extra in synthetic:    []

Real shape: (391, 40)
Synthetic shape: (391, 40)

Missings in real:
Series([], dtype: int64)

Missings in synthetic:
Series([], dtype: int64)

[tobacco_use]
Rows with exactly one 1: 1.0
Rows with zero 1s:       0.0
Rows with >1 1s:         0.0

[primary_tumor]
Rows with exactly one 1: 1.0
Rows with zero 1s:       0.0
Rows with >1 1s:         0.0


### Numerical

In [26]:
def compare_numerical_distributions(real_df, syn_df, numeric_cols):
    """
    Compare numerical variables using mean, std, KS, and Wasserstein.
    """
    rows = []

    for c in numeric_cols:
        x = pd.to_numeric(real_df[c], errors="coerce").dropna()
        y = pd.to_numeric(syn_df[c], errors="coerce").dropna()

        rows.append({
            "variable": c,
            "real_mean": x.mean(),
            "syn_mean": y.mean(),
            "real_std": x.std(),
            "syn_std": y.std(),
            "real_p25": x.quantile(0.25),
            "syn_p25": y.quantile(0.25),
            "real_p50": x.quantile(0.50),
            "syn_p50": y.quantile(0.50),
            "real_p75": x.quantile(0.75),
            "syn_p75": y.quantile(0.75),
            "ks_stat": ks_2samp(x, y).statistic,
            "wasserstein": wasserstein_distance(x, y)
        })

    return pd.DataFrame(rows).sort_values(["ks_stat", "wasserstein"], ascending=False)

In [27]:
num_report = compare_numerical_distributions(df_model, synthetic_raw, numeric_cols)
num_report

,variable,real_mean,syn_mean,real_std,syn_std,real_p25,syn_p25,real_p50,syn_p50,real_p75,syn_p75,ks_stat,wasserstein
10,rs2232698_n_risk_alleles,0.023018,1.000000,0.150152,0.000000,0.000,1.000,0.0,1.00,0.00,1.00,0.976982,0.976982
9,rs6025_n_risk_alleles,0.028133,1.000000,0.165565,0.000000,0.000,1.000,0.0,1.00,0.00,1.00,0.971867,0.971867
7,rs268_n_risk_alleles,0.033248,1.000000,0.179513,0.000000,0.000,1.000,0.0,1.00,0.00,1.00,0.966752,0.966752
8,rs5110_n_risk_alleles,0.171355,0.969309,0.409875,0.873589,0.000,0.000,0.0,1.00,0.00,2.00,0.445013,0.797954
11,rs5985_n_risk_alleles,0.450128,1.219949,0.600806,0.766418,0.000,1.000,0.0,1.00,1.00,2.00,0.398977,0.769821
2,rs11696364_n_risk_alleles,0.199488,0.872123,0.448473,0.890989,0.000,0.000,0.0,1.00,0.00,2.00,0.352941,0.672634
3,rs169713_n_risk_alleles,0.539642,1.168798,0.626622,0.724762,0.000,1.000,0.0,1.00,1.00,2.00,0.340153,0.629156
6,rs2227631_n_risk_alleles,0.928389,1.491049,0.690587,0.563229,0.000,1.000,1.0,2.00,1.00,2.00,0.319693,0.562660
5,rs6003_n_risk_alleles,1.813299,1.695652,0.403100,0.625900,2.000,2.000,2.0,2.00,2.00,2.00,0.084399,0.117647
0,age_cancer_dx,64.797954,65.189258,10.991026,10.694354,57.000,58.000,66.0,66.00,73.50,73.00,0.051151,0.636829


### Categorical

In [28]:
def total_variation_distance(p, q):
    """
    Total variation distance between two discrete distributions.
    """
    idx = sorted(set(p.index).union(set(q.index)))
    p = p.reindex(idx, fill_value=0)
    q = q.reindex(idx, fill_value=0)
    return 0.5 * np.abs(p - q).sum()

In [29]:
def compare_categorical_distributions(real_df, syn_df, categorical_cols):
    """
    Compare proportions of categorical columns.
    """
    rows = []
    tables = {}

    for c in categorical_cols:
        p = real_df[c].value_counts(normalize=True, dropna=False)
        q = syn_df[c].value_counts(normalize=True, dropna=False)

        tvd = total_variation_distance(p, q)
        comp = pd.concat(
            [p.rename("real"), q.rename("synthetic")],
            axis=1
        ).fillna(0)

        rows.append({
            "variable": c,
            "n_categories_real": real_df[c].nunique(dropna=False),
            "n_categories_syn": syn_df[c].nunique(dropna=False),
            "total_variation_distance": tvd
        })

        tables[c] = comp.sort_index()
        
    summary = pd.DataFrame(rows).sort_values("total_variation_distance", ascending=False)
    return summary, tables

In [30]:
cat_cols_eval = categorical_cols.copy()
if TARGET_TYPE == "categorical":
    cat_cols_eval = cat_cols_eval + [TARGET_COL]

cat_summary, cat_tables = compare_categorical_distributions(df_model, synthetic_raw, cat_cols_eval)
cat_summary

,variable,n_categories_real,n_categories_syn,total_variation_distance
5,dyslipidemia,2,2,0.056266
15,performance_status,3,3,0.046036
11,tumor_surgically_removed,2,2,0.040921
14,pTNM_stage,4,4,0.023018
8,previous_ate,2,2,0.023018
18,primary_tumor,7,7,0.020460
17,tobacco_use,3,3,0.020460
4,diabetes_mellitus,2,2,0.020460
19,VTE,2,2,0.017903
16,grade_histological_differentiation,3,3,0.017903


In [31]:
cat_tables["pTNM_stage"]
cat_tables["performance_status"]
cat_tables["tobacco_use"]
cat_tables["primary_tumor"]

,real,synthetic
primary_tumor,,
Bile duct,0.097187,0.099744
Brain,0.010230,0.012788
Colorectal,0.263427,0.278772
Gastric,0.115090,0.115090
NSCLC,0.191816,0.191816
Oesophageal,0.035806,0.033248
Pancreatic,0.286445,0.268542
